<a href="https://colab.research.google.com/github/penajuanmanuel6-hub/automated-etl-pipeline/blob/main/analisis_impactos_sismicos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# SCRIPT INTEGRADO: SQL + PANDAS + PLOTLY PARA ANÁLISIS DE IMPACTOS SÍSMICOS
# ==============================================================================

import sqlite3
import pandas as pd
import plotly.express as px

print("--- 1. CONFIGURACIÓN DE LA BASE DE DATOS Y CARGA DE DATOS (SQL) ---")

# Creamos una base de datos SQLite en memoria
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Creación de tablas relacionales
cursor.execute('''
CREATE TABLE sectores (
    sector_id INTEGER PRIMARY KEY,
    sector_nombre TEXT,
    pib_sectorial_usd REAL
);
''')

cursor.execute('''
CREATE TABLE infraestructuras (
    infraestructura_id INTEGER PRIMARY KEY,
    sector_id INTEGER,
    evento_sismico_id TEXT,
    costo_reconstruccion_usd REAL,
    porcentaje_destruccion REAL,
    empleos_afectados INTEGER,
    FOREIGN KEY (sector_id) REFERENCES sectores (sector_id)
);
''')

# Inserción de datos de sectores
sectores_data = [
    (1, 'Infraestructura Vial', 5000.0),
    (2, 'Vivienda y Habitacional', 8000.0),
    (3, 'Comercio y Turismo', 7000.0),
    (4, 'Industria Manufacturera', 9000.0),
    (5, 'Salud y Educación', 4000.0),
    (6, 'Agricultura', 3000.0)
]
cursor.executemany("INSERT INTO sectores VALUES (?, ?, ?)", sectores_data)

# Inserción de datos de infraestructuras afectadas agrupadas por sector
infra_data = [
    (101, 1, 'SISMO_2026_01', 1200.0, 24.0, 45000),
    (102, 2, 'SISMO_2026_01', 950.0, 11.8, 120000),
    (103, 3, 'SISMO_2026_01', 600.0, 8.5, 85000),
    (104, 4, 'SISMO_2026_01', 450.0, 5.0, 60000),
    (105, 5, 'SISMO_2026_01', 300.0, 7.5, 35000),
    (106, 6, 'SISMO_2026_01', 200.0, 6.6, 50000)
]
cursor.executemany("INSERT INTO infraestructuras VALUES (?, ?, ?, ?, ?, ?)", infra_data)
conn.commit()

print("Base de datos estructurada y poblada correctamente.\n")

print("--- 2. EJECUCIÓN DE LA CONSULTA SQL DE AGREGACIÓN Y PRIORIZACIÓN ---")

# Consulta SQL solicitada para extraer y consolidar métricas de impacto sectorial
query_sql = """
SELECT
    s.sector_nombre AS Sector,
    COUNT(DISTINCT i.infraestructura_id) AS Total_Infraestructuras_Afectadas,
    SUM(i.costo_reconstruccion_usd) AS Impacto_USD_Millones,
    SUM(i.empleos_afectados) AS Empleos_Afectados,
    AVG(i.porcentaje_destruccion) AS Promedio_Destruccion_Pct,
    s.pib_sectorial_usd AS PIB_Sectorial_USD_Millones,
    ROUND((SUM(i.costo_reconstruccion_usd) / s.pib_sectorial_usd) * 100, 2) AS Impacto_Sobre_PIB_Pct,
    CASE
        WHEN (SUM(i.costo_reconstruccion_usd) / s.pib_sectorial_usd) > 0.15 THEN 'Crítica - Prioridad 1'
        WHEN (SUM(i.costo_reconstruccion_usd) / s.pib_sectorial_usd) BETWEEN 0.05 AND 0.15 THEN 'Alta - Prioridad 2'
        ELSE 'Moderada - Prioridad 3'
    END AS Nivel_Prioridad_SQL
FROM
    infraestructuras i
JOIN
    sectores s ON i.sector_id = s.sector_id
WHERE
    i.evento_sismico_id = 'SISMO_2026_01'
GROUP BY
    s.sector_nombre, s.pib_sectorial_usd
ORDER BY
    Impacto_USD_Millones DESC;
"""

# Leemos directamente el resultado de la consulta SQL en un DataFrame de Pandas
df = pd.read_sql_query(query_sql, conn)

# Cerramos la conexión a la base de datos
conn.close()

# Cálculo adicional en Python (Scoring ponderado para refinar la visualización)
df['Score_Prioridad'] = (
    0.5 * (df['Impacto_Sobre_PIB_Pct'] / df['Impacto_Sobre_PIB_Pct'].max()) +
    0.5 * (df['Empleos_Afectados'] / df['Empleos_Afectados'].max())
) * 100

print("\n--- RESULTADO DE LA CONSULTA SQL & MATRIZ ---")
print(df[['Sector', 'Impacto_USD_Millones', 'Empleos_Afectados', 'Impacto_Sobre_PIB_Pct', 'Nivel_Prioridad_SQL']].to_string(index=False))

print("\n--- 3. GENERACIÓN DE GRÁFICO EJECUTIVO INTERACTIVO (PLOTLY) ---")

# Creación de gráfico de burbujas interactivo basado en los datos de SQL
fig = px.scatter(
    df,
    x="Impacto_USD_Millones",
    y="Empleos_Afectados",
    size="Score_Prioridad",
    color="Sector",
    hover_name="Sector",
    text="Sector",
    size_max=50,
    title="<b>Matriz Ejecutiva de Impacto Sísmico y Focalización de Inversión (Vía SQL)</b>",
    labels={
        "Impacto_USD_Millones": "Impacto Financiero (Millones USD)",
        "Empleos_Afectados": "Población / Empleos Afectados"
    }
)

fig.update_traces(textposition='top center')
fig.update_layout(
    template="plotly_white",
    title_font_size=16,
    showlegend=False,
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True),
    height=600
)

# Renderizar el gráfico interactivo en Google Colab
fig.show()

--- 1. CONFIGURACIÓN DE LA BASE DE DATOS Y CARGA DE DATOS (SQL) ---
Base de datos estructurada y poblada correctamente.

--- 2. EJECUCIÓN DE LA CONSULTA SQL DE AGREGACIÓN Y PRIORIZACIÓN ---

--- RESULTADO DE LA CONSULTA SQL & MATRIZ ---
                 Sector  Impacto_USD_Millones  Empleos_Afectados  Impacto_Sobre_PIB_Pct   Nivel_Prioridad_SQL
   Infraestructura Vial                1200.0              45000                  24.00 Crítica - Prioridad 1
Vivienda y Habitacional                 950.0             120000                  11.88    Alta - Prioridad 2
     Comercio y Turismo                 600.0              85000                   8.57    Alta - Prioridad 2
Industria Manufacturera                 450.0              60000                   5.00    Alta - Prioridad 2
      Salud y Educación                 300.0              35000                   7.50    Alta - Prioridad 2
            Agricultura                 200.0              50000                   6.67    Alta - Prior